# Chapter 3: Groups - Examples

Jupyter notebook demonstrating group theory concepts: cyclic groups, element orders, primitive roots, Permutation groups, discrete logarithm, Diffie-Hellman.

## 1. Cyclic groups and element orders

$\mathbb{Z}_7^\times$ is cyclic of order 6. Find: which elements are generators, and the order of any element $g^k$ given by $o(g^k) = o(g)/\gcd(k, o(g))$.

In [ ]:
import math

n = 7
group = [a for a in range(1, n) if math.gcd(a, n) == 1]  # Z_7^{\times}
print(f"Z_{n}^\u00d7 = {group}, order = {len(group)}")

# compute order of each element
for g in group:
    order = 1
    val = g % n
    while val != 1:
        val = val * g % n
        order += 1
    is_gen = " GENERATOR" if order == len(group) else ""
    print(f"order({g}) = {order}{is_gen}")

## 2. o(g^k) = o(g) / gcd(k, o(g))

Verify the formula for each element of $\mathbb{Z}_{7}^\times = \langle 3 \rangle$.

In [ ]:
def element_order(g, n):
    order = 1; val = g % n
    while val != 1:
        val = val * g % n; order += 1
    return order

g = 3  # generator of Z_7^\times, order = 6
o_g = element_order(g, 7)
print(f"o(3) = {o_g}")
for k in range(6):
    el = pow(g, k, 7)
    o_el = element_order(el, 7)
    formula = o_g // math.gcd(k, o_g)
    print(f"o(3^{k}) = o({el}) = {o_el}, formula gives {formula}  {'OK' if o_el==formula else 'MISMATCH'}")

## 3. Primitive roots

Primitive root modulo $n$ exists exactly for $n \in \{2, 4, p^k, 2p^k\}$. Count and find generators of $\mathbb{Z}_{n}^{\times}$.

In [ ]:
def primitive_roots(n):
    group = [a for a in range(1, n) if math.gcd(a, n) == 1]
    order_all = len(group)
    roots = []
    for g in group:
        if element_order(g, n) == order_all:
            roots.append(g)
    return roots

for n in [2, 4, 8, 9, 15, 25]:
    roots = primitive_roots(n)
    print(f"n = {n}: existence = {bool(roots)}, primitive roots = {roots}, count = {len(roots)}")

## 4. Permutation groups and cycle decomposition

Decompose a permutation into disjoint cycles and compute its sign (parity): $+1$ if even, $-1$ if odd.

In [ ]:
def to_cycles(perm):
    """Decompose a permutation into disjoint cycles.
    perm[i] = image of element i (0-indexed)."""
    n = len(perm)
    seen = set()
    cycles = []
    for i in range(n):
        if i not in seen:
            start = i
            cyc = []
            while start not in seen:
                seen.add(start)
                cyc.append(start)
                start = perm[start]
            if len(cyc) > 1:
                cycles.append(cyc)
    return cycles

# sigma in 1-indexed notation: 1->2, 2->4, 3->1, 4->5, 5->3.
# Convert to 0-indexed images: image of i (0-based) = sigma(i+1)-1.
perm = [2, 4, 1, 5, 3]
perm0 = [x - 1 for x in perm]  # images: 0->1, 1->3, 2->0, 3->4, 4->2
cycles = to_cycles(perm0)
print("Cycle decomposition (0-indexed):", cycles)
print("Meaning (1-indexed):", [[c + 1 for c in cyc] for cyc in cycles])
sign = 1
for c in cycles:
    sign *= (-1) ** (len(c) + 1)
print("Lengths: %s, sign = %d (%s permutation)" % ([len(c) for c in cycles], sign, 'even' if sign == 1 else 'odd'))

## 5. Discrete logarithm by brute force (small example)

For a small cyclic group we can compute logarithms by brute force. For actual cryptography the group is chosen large so this is infeasible.

In [ ]:
def discrete_log(g, h, n):
    """Find x with g^x = h mod n (m is the order of g). Brute force."""
    val = 1
    for x in range(n):
        if val == h:
            return x
        val = val * g % n
    return None

p, g = 23, 5   # 5 generates Z_23^\times
h = 19
x = discrete_log(g, h, p)
print(f"log_{g}({h}) mod {p} = {x}, check 5^{x} mod 23 = {pow(g, x, p)}")

## 6. Baby-step giant-step for the DLP

A generic $O(\sqrt{n})$ algorithm for the DLP. Useful for verifying small group logs efficiently.

In [ ]:
import math

def babystep_giantstep(g, h, p, order=None):
    """Solve g^x = h mod p. Returns x."""
    if order is None:
        order = p - 1  # assume g is a generator of Z_p^\times
    m = math.isqrt(order) + 1
    # baby steps: store g^j for j = 0..m-1
    table = {}
    e = 1
    for j in range(m):
        table.setdefault(e, j)
        e = e * g % p
    # giant steps: h * (g^{-m})^i
    factor = pow(g, -m, p)  # g^{-m} mod p
    gamma = h
    for i in range(m):
        if gamma in table:
            return i * m + table[gamma]
        gamma = gamma * factor % p
    return None

p, g = 1000003, 3  # smaller prime, groups not generator - just demo
# Use small p for clarity
p, g = 97, 5
h = 44
x = babystep_giantstep(g, h, p)
print(f"log_{g}({h}) mod {p} = {x}")
print(f"Check: {g}^{x} mod {p} = {pow(g, x, p)}")

## 7. Diffie-Hellman key exchange

The protocol from Section 'The Discrete Logarithm'. Both parties compute the same shared secret $g^{ab} \bmod p$.

In [ ]:
def dh_exchange(p, g, a, b):
    A = pow(g, a, p)   # Alice's public
    B = pow(g, b, p)   # Bob's public
    K_alice = pow(B, a, p)  # shared secret
    K_bob = pow(A, b, p)
    assert K_alice == K_bob
    return A, B, K_alice

p, g = 23, 5
a, b = 6, 15
A, B, K = dh_exchange(p, g, a, b)
print(f"Alice public A = {A}, Bob public B = {B}")
print(f"Shared secret K = {K}")
print(f"Check g^(ab) mod p = {pow(g, a*b, p)}")